In [ ]:
import tkinter as tk
from PIL import Image, ImageTk
import numpy as np

class MiniViewer:
    def __init__(self, width: int, height: int):
        self.root = tk.Tk()
        self.root.title("Viewer")
        self.canvas = tk.Canvas(self.root, width=width, height=height)
        self.canvas.pack()
        self._tk_img = None
        self._running = True
        self.root.protocol("WM_DELETE_WINDOW", self._on_close)

    def update_from_array(self, idx: int, mat: np.ndarray):
        if self._running:
            self.root.after(0, self._render_array, mat.copy())

    def _render_array(self, mat: np.ndarray):
        if not self._running:
            return
        try:
            img = Image.fromarray(mat[..., ::-1])
            self._tk_img = ImageTk.PhotoImage(img)           
            self.canvas.delete("all")                       
            self.canvas.create_image(0, 0, anchor="nw", image=self._tk_img)
        except Exception:
            pass

    def _on_close(self):
        self._running = False
        self.root.destroy()

    def start(self):
        self.root.mainloop()

# SingleViewer

Plays back a sequence of image frames from a `FramesDir` at a given frame rate, invoking a callback on each frame update. Designed to run asynchronously alongside a UI layer such as tkinter.

---

## Construction

The class is instantiated directly via `__init__`:

* **`SingleViewer(frames_dir, fps, on_frame_update)`** — creates a viewer from a `FramesDir` instance. The playback speed is determined by `fps`, and `on_frame_update` is the callback invoked on every frame advance.

---

## Attributes

| Attribute | Type | Description |
|---|---|---|
| `frames_dir` | `FramesDir` | Source of the frame sequence |
| `fps` | `int` | Playback frame rate |
| `on_frame_update` | `Callable[[int, str], None]` | Callback invoked with `(frame_idx, frame_path)` on each frame |

---

## Operations

* **`start(loop?)`** — starts the internal async playback loop and immediately plays. If an external `asyncio` event loop is provided (e.g. when running alongside tkinter), the loop is scheduled via `run_coroutine_threadsafe`. Otherwise `create_task` is used, assuming an active loop in the current context.

* **`play()`** — resumes playback by setting the internal play event.

* **`pause()`** — suspends playback. The loop remains alive but waits until `play()` is called again.

* **`stop()`** — terminates the playback loop, cancels the asyncio task and resets the internal state.

* **`set_frame_idx(frame_idx)`** — jumps to a specific frame. Temporarily pauses playback during the seek and resumes automatically after.

* **`set_frame_start(frame_start)`** — sets the start boundary of the playback range.

* **`set_frame_end(frame_end)`** — sets the end boundary of the playback range.

---

## Playback Loop Behavior

The internal `_run` coroutine iterates over the frame range in a continuous loop. On each iteration it:

1. Waits for the play event to be set
2. Invokes `on_frame_update(frame_idx, frame_path)`
3. Sleeps for `1 / fps` seconds before advancing to the next frame

When the end of the range is reached, playback loops back to `frame_start`.

---

## Threading Model

`SingleViewer` supports two execution contexts:

| Context | How to start |
|---|---|
| Inside `async def` / Jupyter | `viewer.start()` |
| External thread (e.g. tkinter) | `viewer.start(loop=loop)` |

When used with a UI, the `on_frame_update` callback must schedule any UI update back onto the main thread (e.g. via `root.after(0, ...)`), as the callback is invoked from the asyncio thread.

---

## Dependencies

| Class | Role |
|---|---|
| `FramesDir` | Source of the frame sequence |
| `asyncio.Event` | Controls play/pause state |

In [3]:
import asyncio
import threading
import movixpy.io as mvi
from movixpy.viewer import SingleViewer

video_file = mvi.VideoFile.from_video(r".\data\01.mp4")
frames_dir = await mvi.FramesDir.from_video(r".\data\01_view", video_file)

viewer = MiniViewer(video_file.width, video_file.height)

single_viewer = SingleViewer(
    frames_dir = frames_dir,
    fps = video_file.fps,
    on_frame_update=viewer.update_from_path
)

loop = asyncio.new_event_loop()
threading.Thread(target=loop.run_forever, daemon=True).start()

single_viewer.start(loop=loop)

viewer.start()

single_viewer.stop()
loop.stop()

# MultiViewer

In [ ]:
import asyncio
import threading
import movixpy.io as mvi

from movixpy.viewer import MultiViewer

video_file1 = mvi.VideoFile(r".\data\01.mp4")
video_file2 = mvi.VideoFile(r".\data\02.mp4")

motion_frames_dir1 = await mvi.MotionFramesDir.from_video(r".\data\mov\01", video_file1, key_positions=[])
motion_frames_dir1.add(mvi.KeyPosition(0, (0, 0, 500, 500), 0))
motion_frames_dir1.add(mvi.KeyPosition(len(motion_frames_dir1) - 1, (1000, 1000, 500, 500), 0))

motion_frames_dir2 = await mvi.MotionFramesDir.from_video(r".\data\mov\02", video_file2, key_positions=[])
motion_frames_dir2.add(mvi.KeyPosition(0, (-500, -500, 500, 500), 90))
motion_frames_dir2.add(mvi.KeyPosition(len(motion_frames_dir2) - 1, (500, 500, 500, 500), 90))

viewer = MiniViewer(1920, 1080)

motion_frames_dirs = [motion_frames_dir1, motion_frames_dir2]
multi_viewer = MultiViewer(
    motion_frames_dirs, 
    video_file1.fps,
    1920, 
    1080, 
    (255, 255, 255),
    viewer.update_from_array
)

loop = asyncio.new_event_loop()
threading.Thread(target=loop.run_forever, daemon=True).start()

multi_viewer.start(loop=loop)

viewer.start()

multi_viewer.stop()
loop.stop()